# Atelier Preparation de Donnees Textuelles

## Partie 1 - Exploration du corpus

### 1) Chargement des donnees CSV

In [ ]:
import pandas as pd

df = pd.read_csv("../data/smart_reviews_raw.csv", encoding="utf-8")
df.head()

### 2) et 3) Nombre d'avis et nombre de colonnes

In [ ]:
n_avis, n_colonnes = df.shape
print(f"Nombre d'avis : {n_avis}")
print(f"Nombre de colonnes : {n_colonnes}")

### 4) Type de chaque colonne

In [ ]:
df.dtypes

### 5) Valeurs manquantes

In [ ]:
df.isna().sum()

### 6) Identifier quelques types de texte

In [ ]:
import re

texte = df["texte"]

exemples = {
    "normal": texte[texte.str.len().between(30, 60) & texte.notna()].iloc[0],
    "vide": texte[texte.isna()].index[0] if texte.isna().any() else None,
    "url": texte[texte.str.contains("http", na=False)].iloc[0],
    "mention": texte[texte.str.contains("@", na=False)].iloc[0],
    "hashtag": texte[texte.str.contains("#", na=False)].iloc[0],
    "emoji": texte[texte.str.contains(r"[\U0001F300-\U0001FAFF]", na=False, regex=True)].iloc[0],
    "ponctuation": texte[texte.str.contains(r"[!?]{2,}", na=False, regex=True)].iloc[0],
    "majuscules": texte[texte.str.isupper().fillna(False)].iloc[0],
    "repetition": texte[texte.str.contains(r"(.)\1{2,}", na=False, regex=True)].iloc[0],
}

for type_texte, exemple in exemples.items():
    print(f"{type_texte:12s}: {exemple}")

### 7) Longueur des textes

In [ ]:
df["longueur"] = df["texte"].str.len()

stats_longueur = df["longueur"].describe()
print(f"Longueur minimale : {df['longueur'].min()}")
print(f"Longueur maximale : {df['longueur'].max()}")
print(f"Longueur moyenne  : {df['longueur'].mean():.2f}")
print(f"Longueur mediane  : {df['longueur'].median()}")
print(f"Q1 : {df['longueur'].quantile(0.25)}")
print(f"Q3 : {df['longueur'].quantile(0.75)}")
stats_longueur

### 8) Visualisation de la distribution de la longueur des avis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["longueur"].dropna(), bins=40, color="steelblue")
axes[0].set_title("Distribution de la longueur des avis")
axes[0].set_xlabel("Longueur (caracteres)")
axes[0].set_ylabel("Frequence")

axes[1].boxplot(df["longueur"].dropna(), vert=True)
axes[1].set_title("Boxplot de la longueur des avis")
axes[1].set_ylabel("Longueur (caracteres)")

plt.tight_layout()
plt.show()

**a) Existe-t-il des textes anormalement longs ?**

Oui. La moyenne se situe autour de 49 caracteres et le 3e quartile autour de 56, mais le maximum atteint plus de 600 caracteres. Ces valeurs tres eloignees de la distribution principale (visibles comme points isoles au-dessus de la moustache superieure du boxplot) constituent des outliers a examiner : ils peuvent correspondre a des avis tres detailles, du texte duplique/spam, ou du contenu concatene par erreur lors de la collecte.

**b) Existe-t-il beaucoup de textes tres courts ?**

Oui, une partie non negligeable des avis a une longueur proche du minimum observe (2 caracteres), ce qui correspond a des textes quasi vides ou peu informatifs (ex: un seul mot, une ponctuation). Ces textes tres courts risquent d'apporter peu de signal pour la classification de sentiment et devront etre surveilles lors du nettoyage.

### 9) Detection des textes vides

In [ ]:
textes_vides = df[df["texte"].isna() | (df["texte"].str.strip() == "")]
print(f"Nombre de textes vides : {len(textes_vides)}")
textes_vides[["id_avis", "texte"]]